# 02 — Chunking & INCEpTION export

**Project:** Misurare la creatività linguistica: sorpresa ed entropia come strumenti di analisi stilistica (Premio di ricerca Dino Buzzetti, AIUCD)

This notebook orchestrates phase 02. It contains no logic of its own — it calls two single-responsibility modules in `scripts/` and inspects their output:

1. **`build_canonical`** — runs Stanza once per normalized text → canonical CoNLL-U (single source of truth: tokenization + sentence segmentation + POS, with **global** character offsets in MISC).
2. **`run`** (chunk_sampling) — samples 5 chunks per text (one per quintile, blind to surprisal), accumulating whole sentences to ≥ 500 surface words, and writes the four artefacts for annotation.
3. **`collapse_mwt`** — post-processes the `inception/` documents, collapsing Stanza MWTs to surface orthographic tokens (annotation unit = orthographic word, aligned with GePpeTto's BPE surface form). The canonical files are left MWT-expanded.

**Prerequisite:** `pip install stanza` (Italian models download on first run). Stanza runs locally on CPU — no API, no credits.

**Reproducibility parameters** (also recorded in `docs/decisions_log_2_chunking_annotation.md`): `SEED = 20260622`, `MIN_WORDS = 500`, `CONTEXT_SENTS = 2`.

## Setup

Locate the repository root (the folder containing `scripts/`) and resolve the data paths, so the notebook runs whether Jupyter is launched from the repo root or from `notebooks/`.

In [2]:
import sys, json
from pathlib import Path
from collections import Counter

# Find the repo root: walk up until we see a `scripts/` folder.
ROOT = Path.cwd()
while not (ROOT / 'scripts').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# Paths
NORMALIZED = ROOT / 'data' / 'normalized'
CANONICAL  = ROOT / 'data' / 'chunks' / 'canonical'
OUT        = ROOT / 'data' / 'chunks'
INCEPTION  = OUT / 'inception'

# Reproducibility parameters (see decisions log)
SEED = 20260622
MIN_WORDS = 500
CONTEXT_SENTS = 2

n_txt = len(list(NORMALIZED.glob('*.txt'))) if NORMALIZED.is_dir() else 'MISSING'
print('repo root :', ROOT)
print('normalized:', NORMALIZED, '->', n_txt, 'txt')
print('canonical :', CANONICAL)
print('output    :', OUT)

repo root : c:\Users\silvi\OneDrive\Università\PROGETTI\AIUCD Buzzetti\Project\surprisal-entropy-italian-literature
normalized: c:\Users\silvi\OneDrive\Università\PROGETTI\AIUCD Buzzetti\Project\surprisal-entropy-italian-literature\data\normalized -> 11 txt
canonical : c:\Users\silvi\OneDrive\Università\PROGETTI\AIUCD Buzzetti\Project\surprisal-entropy-italian-literature\data\chunks\canonical
output    : c:\Users\silvi\OneDrive\Università\PROGETTI\AIUCD Buzzetti\Project\surprisal-entropy-italian-literature\data\chunks


## Step 1 — Build canonical CoNLL-U (Stanza)

Runs once per text on the **full** normalized string, so `start_char`/`end_char` are global offsets into the normalized text. This step is slow and downloads the Italian models on first run.

**Idempotency:** texts whose `.conllu` already exists are skipped. If you have corrected a normalized text, its old canonical is now **stale** — delete that single `.conllu` (or pass `overwrite=True`) before re-running, otherwise the guard will skip it. Log every regeneration alongside the normalization change that motivated it.

In [2]:
from scripts.build_canonical import build_canonical

canonical_files = build_canonical(NORMALIZED, CANONICAL)   # add overwrite=True to force a full rebuild
print('canonical files:', len(canonical_files))

C:\Users\silvi\AppData\Roaming\Python\Python311\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.0.4) or chardet (7.4.3)/charset_normalizer (3.2.0) doesn't match a supported version!
  warnings.warn(
C:\Users\silvi\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\silvi\AppData\Roaming\Python\Python311\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


skip (exists): 1869_tarchetti_fosca.conllu
skip (exists): 1878_dossi_desinenza_a.conllu
skip (exists): 1881_verga_malavoglia.conllu
skip (exists): 1889_d_annunzio_piacere.conllu
skip (exists): 1891_serao_paese_cuccagna.conllu
skip (exists): 1895_fogazzaro_piccolo_mondo_antico.conllu
skip (exists): 1904_deledda_cenere.conllu
skip (exists): 1919_tozzi_occhi_chiusi.conllu
skip (exists): 1920_palazzeschi_il_codice_perela.conllu
skip (exists): 1925_pirandello_uno_nessuno_centomila.conllu
skip (exists): 1926_svevo_coscienza_zeno.conllu

canonical files: 11  | in c:\Users\silvi\OneDrive\Università\PROGETTI\AIUCD Buzzetti\Project\surprisal-entropy-italian-literature\data\chunks\canonical
canonical files: 11


## Step 2 — Sample chunks and export

For each text: 5 chunks, one per quintile (stratified by surface-word position), random anchor constrained to `word_start ≤ quintile_end − MIN_WORDS`, target = whole sentences accumulated to ≥ MIN_WORDS (option (i): minimal tail overflow, never an intra-sentence cut). Sampling is **blind to surprisal** by design.

Writes under `data/chunks/`:
- `inception/{chunk_id}.conllu` — 55 target-only documents (import as 55 documents into **one** INCEpTION project)
- `metadata.json` — per-chunk records; single source for the readable passage
- `offset_map.csv` — tool-independent realignment map
- `appendix.md` — human-readable passages (derived view)

In [3]:
from scripts.chunk_sampling import run

chunks, flags = run(CANONICAL, OUT, seed=SEED,
                    min_words=MIN_WORDS, context_sents=CONTEXT_SENTS)
print(f'\n{len(chunks)} chunks generated')

chunks: 55  | seed=20260622

55 chunks generated


## Step 3 — Collapse MWT (surface tokens for INCEpTION)

Post-processes the `inception/{chunk_id}.conllu` files in place: each Stanza MWT range line (`N-M`) is kept as a single surface token, its expanded syntactic words dropped, with per-sentence renumbering. Touches only the INCEpTION-facing derivation; the canonical files stay MWT-expanded. Must run **after** Step 2, which rewrites `inception/` on every execution.

In [4]:
from scripts.collapse_mwt import collapse_mwt

collapsed = collapse_mwt(INCEPTION)   # in-place on the 55 inception/ files
print(f'{len(collapsed)} files collapsed to surface tokens')

55 files collapsed to surface tokens


## Step 4 — Inspect flags and sanity checks

`flags` reports non-blocking guards: quintiles too short to host a chunk, or overlaps between adjacent chunks. On the 11 long novels these should be empty.

In [5]:
print('flags:', flags if flags else 'none')

# Chunks per text — expect 5 each
per_text = Counter(c['text_id'] for c in chunks)
for t, n in sorted(per_text.items()):
    print(f'  {t}: {n} chunks')

# Target length distribution (all >= MIN_WORDS by construction)
wc = [c['target_word_count'] for c in chunks]
if wc:
    print(f'\ntarget words - min {min(wc)}, max {max(wc)}, mean {sum(wc)//len(wc)}')

flags: none
  1869_tarchetti_fosca: 5 chunks
  1878_dossi_desinenza_a: 5 chunks
  1881_verga_malavoglia: 5 chunks
  1889_d_annunzio_piacere: 5 chunks
  1891_serao_paese_cuccagna: 5 chunks
  1895_fogazzaro_piccolo_mondo_antico: 5 chunks
  1904_deledda_cenere: 5 chunks
  1919_tozzi_occhi_chiusi: 5 chunks
  1920_palazzeschi_il_codice_perela: 5 chunks
  1925_pirandello_uno_nessuno_centomila: 5 chunks
  1926_svevo_coscienza_zeno: 5 chunks

target words - min 500, max 580, mean 517


### Preview a passage

Visual check of one chunk: the target is delimited by ⟦ … ⟧, with read-only context on either side.

In [6]:
meta = json.loads((OUT / 'metadata.json').read_text(encoding='utf-8'))
c = meta[0]
print(c['chunk_id'], '| quinto', c['quinto'], '|', c['target_word_count'], 'parole\n')
print(c['passage'][:800], '...')

1869_tarchetti_fosca_q1 | quinto 1 | 502 parole

Non di amore, no; non amava ancora, non ne sperava; ma assetato di conforti, di compianto, di lacrime.  Avrei desiderato una donna, non per chiederle le sue carezze, ma per piangere sul suo seno. ⟦L'uomo è più profondo nell'amore, la donna nella tenerezza; si piange meglio sul seno di una donna.  Non so se gli altri uomini abbiano sùbiti abbandoni, sùbiti impeti, sùbite risoluzioni come ho io.  In me vi è nulla di lento, di ordinato, di normale.  La mia è una natura a molle, a sbalzi; una natura sempre alterata.  Le scrissi, e le gettai dal balcone un biglietto contenente queste sole parole:  «Io sono infelice, io sono malato, io soffro».  Il biglietto cadde a' suoi piedi.  Essa lo vide, esitò un istante, poi si curvò, lo raccolse, e fuggì nella sua camera.  Non ricomparve più lungo il gio ...


## Outputs and next steps

The four artefacts are in `data/chunks/`. Next:

1. Create **one** INCEpTION project and import the 55 files in `data/chunks/inception/` as 55 documents. **Before importing all 55**, import a single chunk and verify that the token count in INCEpTION matches the number of surface tokens in that `.conllu` file — this confirms the collapse and INCEpTION's CoNLL-U reader agree on the token grid.
2. Use `appendix.md` (or `metadata.json`) as the read-only **bidirectional** context while annotating; only target tokens are annotated in INCEpTION.
3. Configure the **ordinal markedness scale** as a custom span layer (still to be defined — see open items in the decisions log).

Regenerable outputs under `data/chunks/` may be `.gitignore`d: they are deterministically reproducible from the canonical corpus + seed.